# Swarm Orchestration


In [ ]:
# !pip install autogen --upgrade

Hi and welcome back! In this video, let's implement swarm orchestration on the following problem statement.

# Problem Statement
1. **Axel Limited's Objective**: The company aims to expand into new markets to achieve long-term growth and become an industry giant.

2. **Management's Goal**: Top management wants to discuss the expansion strategy and gather diverse perspectives from all members.

3. **Agentic System**: Simulate a collaborative discussion where each agent represents a member of the top management, providing unique perspectives independently, with smooth and logical conversation transitions.


In [ ]:
import os
import random
import autogen
from autogen import (
ON_CONDITION,
SwarmAgent,
initiate_swarm_chat
)
from autogen import UserProxyAgent, AfterWorkOption
from IPython.display import display, Markdown

In [ ]:
# load env variables
from dotenv import load_dotenv
load_dotenv('/Users/admin/Desktop/AutoGen/Module 1/.env')

In [ ]:
config_list_1 = {
    "config_list": [{"model": "gpt-4o-mini", "temperature": 0.2, "cache_seed": None}]
}
config_list_2 = {
    "config_list": [{"model": "gpt-4o-mini", "temperature": 0.4, "cache_seed": None}]
}

In [ ]:
ceo = SwarmAgent(
    'CEO',
    system_message="You are the CEO (Chief Executive Officer), responsible for overseeing the overall vision, strategy, and alignment of the organization. Focus on high-level goals and strategic priorities while ensuring all functions are aligned.",
    llm_config= config_list_1,
    human_input_mode="NEVER",
)

In [ ]:
cmo = SwarmAgent(
    'CMO',
    system_message="You are the CMO (Chief Marketing Officer), focusing on marketing strategies, brand positioning, and customer engagement. Provide insights into market trends, competitive positioning, and campaign performance.",
    llm_config= config_list_2,
    human_input_mode="NEVER",
)

In [ ]:
cto = SwarmAgent(
    'CTO',
    system_message="You are the CTO (Chief Technical Officer), focused on leveraging technology to improve the customer experience and drive satisfaction. Share insights on how technical solutions can enhance customer-centric initiatives and propose strategies for using technology to optimize the overall experience.",
    llm_config= config_list_2,
    human_input_mode="NEVER",
)

In [ ]:
coo = SwarmAgent(
    'COO',
    system_message="You are the COO (Chief Operating Officer), focusing on operational efficiency, resource management, and process optimization. Provide insights into operational risks, improvements, and performance metrics.",
    llm_config= config_list_2,
    human_input_mode="NEVER",
)

In [ ]:
cfo = SwarmAgent(
    'CFO',
    system_message="You are the CFO (Chief Financial Officer), responsible for financial planning, risk management, and fiscal strategy. Share insights on financial forecasts, budget allocation, and cost management.",
    llm_config= config_list_1,
    human_input_mode="NEVER",
)

In [ ]:
user = UserProxyAgent(
    name="User",
    system_message="Human user interacting with the executive team to discuss company strategies and decisions.",
    code_execution_config={"work_dir":"autogen", "use_docker":False},
    human_input_mode="ALWAYS",
)

In [ ]:
# CEO can hand off to other executives based on their expertise
ceo.register_hand_off(
    [
        ON_CONDITION(cmo, "For marketing strategies and customer engagement"),
        ON_CONDITION(cto, "For technology-related decisions that impact overall strategy"),
        ON_CONDITION(coo, "For operational efficiency and process improvements"),
    ]
)

# CMO can hand off to CEO for strategic alignment or to CTO for technical dependencies in marketing
cmo.register_hand_off(
    [
        ON_CONDITION(ceo, "For strategic alignment of marketing objectives with company goals"),
        ON_CONDITION(cto, "For technology solutions supporting marketing campaigns"),
    ]
)

# CTO can hand off to CEO for overarching tech strategies or COO for implementation feasibility
cto.register_hand_off(
    [
        ON_CONDITION(ceo, "For high-level technology strategy alignment"),
        ON_CONDITION(coo, "For operational feasibility of technical implementations"),
    ]
)

# COO can hand off to CEO for strategic decisions or CFO for financial implications
coo.register_hand_off(
    [
        ON_CONDITION(ceo, "For strategic decisions involving operations"),
        ON_CONDITION(cfo, "For budget or resource allocation concerns"),
    ]
)

# CFO can hand off to CEO for financial alignment or COO for operational cost management
cfo.register_hand_off(
    [
        ON_CONDITION(ceo, "For aligning financial strategy with company goals"),
        ON_CONDITION(coo, "For managing operational costs and budgets"),
    ]
)


In [ ]:
# # Initializing swarm chat with the User 
chat_history, context_variables, last_agent = initiate_swarm_chat(
    initial_agent=ceo,  
    agents=[ceo, cto, coo, cfo, cmo],  
    user_agent= user,
    messages="The company is planning to expand into new markets. Share your perspectives on opportunities, risks, and strategic considerations to ensure successful execution.",
    after_work=AfterWorkOption.SWARM_MANAGER, 
)

In [ ]:
# Ensure that the 'chat_history' attribute is present and is a list
if hasattr(chat_history, 'chat_history') and isinstance(chat_history.chat_history, list):
    formatted_content = []
    # Iterate through the messages and create a formatted Markdown string
    for message in chat_history.chat_history:
        
        if 'content' in message:
            name = message.get('name', 'Unknown')  
            role = message.get('role', 'Unknown')  
            formatted_content.append(f"### {name} ({role}):\n\n{message['content']}")
    
    if formatted_content:
        display(Markdown("\n\n".join(formatted_content)))
    else:
        print("No valid messages found in the chat history.")
else:
    print("Chat history does not contain expected format or messages.")
